<a href="https://colab.research.google.com/github/abcdofbigdata/agents/blob/main/anthropic/004_Controlling_Output_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 4.3 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('aws_access_key')
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('aws_secret_access_key')

In [3]:
import boto3

In [4]:
client = boto3.client("bedrock-runtime", region_name="us-west-2")
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": [{"text": text}]}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": [{"text": text}]}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    response = client.converse(**params)

    return response["output"]["message"]["content"][0]["text"]

In [5]:
messages = []

add_user_message(messages, "Is coffee or  tea better for breakfast?")
add_assistant_message(messages, "They are the same because")

chat(messages)

" of Gödel's incompleteness theorem."

In [6]:
messages = []

add_user_message(messages, "Count from 1 to 10")

chat(messages, stop_sequences=["5", "3, 4"])

"Here's the count from 1 to 10:\n\n1\n2\n3\n4\n"

In [7]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n'

##### Exercise!

- Use message prefilling and stop sequences _only_ to get three different commands in a single response
- There shouldn't be any comments or explanation
- Hint: message prefilling isn't limited to just characters like ```


In [8]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(
    messages,
    "Here are all three commands in a single block without any comments:\n```bash",
)

text = chat(messages, stop_sequences=["```"])
text.strip()

'aws s3 ls\naws ec2 describe-instances\naws lambda list-functions'